# Merge 10 TypePro shards and publish the final dataset

Settings: **Internet ON**, accelerator **None/CPU**. Run under
`duyvu1105` so the automatic host identity can publish
the final Dataset. Optional `TYPEPRO_FINAL_USERNAME/KEY` Secrets may
override it. Shards owned by the final account remain private; shards
owned by the second account are public so they can be attached as
notebook inputs. Run only after all `10` shard Datasets complete.


In [ ]:
SHARD_COUNT = 10
REPOSITORY = 'https://github.com/duyvu1105/TypePro.git'
BRANCH = 'main'
EXPECTED_DATASET_OWNER = 'duyvu1105'
SHARD_SOURCE_CONFIG = [{'label': 'runner_a', 'secret_prefix': 'TYPEPRO_RUNNER_A', 'owner': 'duyvu1105', 'shards': [0, 1, 2, 3, 4], 'public_dataset': False}, {'label': 'runner_b', 'secret_prefix': 'TYPEPRO_RUNNER_B', 'owner': 'duymign', 'shards': [5, 6, 7, 8, 9], 'public_dataset': True}]
MERGE_DATASETS = [{'dataset_id': 'duyvu1105/typepro-build-shard-00', 'logical_shard_index': 0, 'part_index': 0, 'part_count': 1, 'shard_index': 0, 'shard_count': 10, 'public_dataset': False}, {'dataset_id': 'duyvu1105/typepro-build-shard-01', 'logical_shard_index': 1, 'part_index': 0, 'part_count': 1, 'shard_index': 1, 'shard_count': 10, 'public_dataset': False}, {'dataset_id': 'duyvu1105/typepro-build-shard-02', 'logical_shard_index': 2, 'part_index': 0, 'part_count': 2, 'shard_index': 2, 'shard_count': 20, 'public_dataset': False}, {'dataset_id': 'duyvu1105/typepro-build-shard-12', 'logical_shard_index': 2, 'part_index': 1, 'part_count': 2, 'shard_index': 12, 'shard_count': 20, 'public_dataset': False}, {'dataset_id': 'duyvu1105/typepro-build-shard-03', 'logical_shard_index': 3, 'part_index': 0, 'part_count': 3, 'shard_index': 3, 'shard_count': 30, 'public_dataset': False}, {'dataset_id': 'duyvu1105/typepro-build-shard-13', 'logical_shard_index': 3, 'part_index': 1, 'part_count': 3, 'shard_index': 13, 'shard_count': 30, 'public_dataset': False}, {'dataset_id': 'duyvu1105/typepro-build-shard-23', 'logical_shard_index': 3, 'part_index': 2, 'part_count': 3, 'shard_index': 23, 'shard_count': 30, 'public_dataset': False}, {'dataset_id': 'duyvu1105/typepro-build-shard-04', 'logical_shard_index': 4, 'part_index': 0, 'part_count': 1, 'shard_index': 4, 'shard_count': 10, 'public_dataset': False}, {'dataset_id': 'duymign/typepro-build-shard-05', 'logical_shard_index': 5, 'part_index': 0, 'part_count': 1, 'shard_index': 5, 'shard_count': 10, 'public_dataset': True}, {'dataset_id': 'duymign/typepro-build-shard-06', 'logical_shard_index': 6, 'part_index': 0, 'part_count': 1, 'shard_index': 6, 'shard_count': 10, 'public_dataset': True}, {'dataset_id': 'duymign/typepro-build-shard-07', 'logical_shard_index': 7, 'part_index': 0, 'part_count': 2, 'shard_index': 7, 'shard_count': 20, 'public_dataset': True}, {'dataset_id': 'duymign/typepro-build-shard-17', 'logical_shard_index': 7, 'part_index': 1, 'part_count': 2, 'shard_index': 17, 'shard_count': 20, 'public_dataset': True}, {'dataset_id': 'duymign/typepro-build-shard-08', 'logical_shard_index': 8, 'part_index': 0, 'part_count': 1, 'shard_index': 8, 'shard_count': 10, 'public_dataset': True}, {'dataset_id': 'duymign/typepro-build-shard-09', 'logical_shard_index': 9, 'part_index': 0, 'part_count': 3, 'shard_index': 9, 'shard_count': 30, 'public_dataset': True}, {'dataset_id': 'duymign/typepro-build-shard-19', 'logical_shard_index': 9, 'part_index': 1, 'part_count': 3, 'shard_index': 19, 'shard_count': 30, 'public_dataset': True}, {'dataset_id': 'duymign/typepro-build-shard-29', 'logical_shard_index': 9, 'part_index': 2, 'part_count': 3, 'shard_index': 29, 'shard_count': 30, 'public_dataset': True}]
FINAL_DATASET_SLUG = "typepro-python-generative"
SEED = 13

import json
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()

def optional_secret(name):
    try:
        value = secrets.get_secret(name)
    except Exception:
        return None
    value = value.strip() if value else ""
    return value or None

final_username = optional_secret("TYPEPRO_FINAL_USERNAME")
final_key = optional_secret("TYPEPRO_FINAL_KEY")
if bool(final_username) != bool(final_key):
    raise RuntimeError(
        "TYPEPRO_FINAL_USERNAME and TYPEPRO_FINAL_KEY must both be present"
    )
use_explicit_credential = bool(final_username and final_key)
if not use_explicit_credential:
    final_username = EXPECTED_DATASET_OWNER
if final_username.casefold() != EXPECTED_DATASET_OWNER.casefold():
    raise RuntimeError(
        f"Final credential belongs to {final_username!r}, expected "
        f"{EXPECTED_DATASET_OWNER!r}"
    )
SHARD_SOURCES = SHARD_SOURCE_CONFIG
FINAL_SOURCE = {
    "label": "final_owner",
    "owner": EXPECTED_DATASET_OWNER,
    "username": final_username,
    "key": final_key,
}
# The publisher validates Dataset ownership using KAGGLE_USERNAME.
# This is safe to set for host-token authentication: it contains only
# the public username and does not replace Kaggle's access token.
os.environ["KAGGLE_USERNAME"] = final_username
AUTH_ROOT = Path("/kaggle/working/typepro_merge_auth")
AUTH_ROOT.mkdir(parents=True, exist_ok=True)

def use_credential(source):
    if not use_explicit_credential:
        return
    auth_config_dir = AUTH_ROOT / source["label"]
    auth_config_dir.mkdir(parents=True, exist_ok=True)
    os.environ["KAGGLE_CONFIG_DIR"] = str(auth_config_dir)
    os.environ["KAGGLE_USERNAME"] = source["username"]
    os.environ["KAGGLE_KEY"] = source["key"]
    os.environ.pop("KAGGLE_API_TOKEN", None)

os.environ["PYTHONUNBUFFERED"] = "1"
print({
    "shard_sources": [
        {"owner": source["owner"], "shards": source["shards"]}
        for source in SHARD_SOURCES
    ],
    "merge_datasets": [item["dataset_id"] for item in MERGE_DATASETS],
    "final_dataset_owner": EXPECTED_DATASET_OWNER,
    "authentication_source": (
        "explicit Kaggle Secret"
        if use_explicit_credential
        else "Kaggle notebook host"
    ),
    "credentials_printed": False,
})

REPO_DIR = Path("/kaggle/working/TypePro")
INPUT_ROOT = Path("/kaggle/input")
EXTRACTED_INPUT_ROOT = Path("/kaggle/working/extracted_shard_inputs")
MERGED_BUILD = Path("/kaggle/working/typepro_build")
FINAL_DIR = Path("/kaggle/working/typepro_python_generative")

def run(command, cwd=None):
    print("+", " ".join(map(str, command)), flush=True)
    subprocess.run([str(value) for value in command], cwd=cwd, check=True)


## Clone code and install dependencies


In [ ]:
if not REPO_DIR.exists():
    run(["git", "clone", "--branch", BRANCH, "--single-branch", REPOSITORY, REPO_DIR])
PIPELINE_DIR = REPO_DIR / "codet5p_type_retrieval"
run([sys.executable, "-m", "pip", "install", "-q", "-U", "-r", PIPELINE_DIR / "requirements-build.txt"])
if use_explicit_credential:
    run([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "kaggle==1.7.4.2"])
else:
    run([sys.executable, "-m", "pip", "install", "-q", "-U", "kaggle"])

identity_code = "\n".join([
    "import json",
    "import kaggle",
    "values = getattr(kaggle.api, 'config_values', {})",
    "print('TYPEPRO_KAGGLE_IDENTITY=' + json.dumps({",
    "    'username': values.get('username'),",
    "    'auth_method': values.get('auth_method') or 'LEGACY_API_KEY',",
    "}))",
])
identity_result = subprocess.run(
    [sys.executable, "-c", identity_code],
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
identity_prefix = "TYPEPRO_KAGGLE_IDENTITY="
identity_line = next(
    (
        line for line in (identity_result.stdout or "").splitlines()
        if line.startswith(identity_prefix)
    ),
    None,
)
if identity_result.returncode or identity_line is None:
    raise RuntimeError(
        f"Kaggle final-owner authentication failed: {identity_result.stdout}"
    )
identity = json.loads(identity_line[len(identity_prefix):])
authenticated_username = (identity.get("username") or "").strip()
if authenticated_username.casefold() != EXPECTED_DATASET_OWNER.casefold():
    raise RuntimeError(
        f"Kaggle authenticated as {authenticated_username!r}, expected "
        f"final owner {EXPECTED_DATASET_OWNER!r}"
    )
print({
    "authenticated_final_owner": authenticated_username,
    "authentication_method": identity.get("auth_method"),
})


## Validate all attached shard Dataset inputs


In [ ]:
EXTRACTED_INPUT_ROOT.mkdir(parents=True, exist_ok=True)
attached_builds = {}

def register_manifest(marker_path):
    marker = json.loads(marker_path.read_text(encoding="utf-8"))
    coordinates = (marker.get("shard_index"), marker.get("shard_count"))
    attached_builds.setdefault(coordinates, []).append(
        (marker_path.parent, marker)
    )

# Kaggle may normalize or suffix mounted directory names. Resolve each
# input by its validated manifest coordinates instead of guessing a
# /kaggle/input/<dataset-slug> path.
for marker_path in INPUT_ROOT.rglob("shard_manifest.json"):
    register_manifest(marker_path)

for archive_number, archive in enumerate(
    INPUT_ROOT.rglob("typepro_build_shard_*.zip")
):
    if list(archive.parent.rglob("shard_manifest.json")):
        continue
    target = EXTRACTED_INPUT_ROOT / f"archive_{archive_number:02d}"
    target.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as bundle:
        bundle.extractall(target)
    extracted_markers = list(target.rglob("shard_manifest.json"))
    if len(extracted_markers) != 1:
        raise RuntimeError(
            f"Cannot uniquely locate manifest in attached archive {archive}: "
            f"{extracted_markers}"
        )
    register_manifest(extracted_markers[0])

shard_builds = []
for item in MERGE_DATASETS:
    dataset_id = item["dataset_id"]
    index = item["shard_index"]
    expected_count = item["shard_count"]
    candidates = attached_builds.get((index, expected_count), [])
    if len(candidates) != 1:
        raise RuntimeError(
            f"{dataset_id}: expected one attached build for "
            f"{index}/{expected_count}, found {candidates}; "
            f"mounted={sorted(str(path) for path in INPUT_ROOT.iterdir())}"
        )
    build, marker = candidates[0]
    if (
        marker["missing_projects"]
        or marker.get("selected_projects", 0) <= 0
        or marker.get("attempted_projects") != marker.get("selected_projects")
    ):
        raise RuntimeError(f"Invalid/incomplete shard marker: {marker}")
    required = [
        build / "metadata" / "split_manifest.json",
        build / "raw_slices",
        build / "project_status",
        build / "project_kb",
    ]
    missing = [str(path) for path in required if not path.exists()]
    if missing:
        raise RuntimeError(f"{dataset_id}: missing merge inputs: {missing}")
    shard_builds.append(build)
    print(f"Validated {dataset_id}: {marker['attempted_projects']} projects")
print("All shard build directories:", [str(path) for path in shard_builds])


## Merge shard outputs


In [ ]:
merge_script = PIPELINE_DIR / "merge_shards.py"
run([
    sys.executable, "-u", merge_script,
    "--shard-build-dirs", *shard_builds,
    "--work-dir", MERGED_BUILD,
])


## Finalize generative train/validation/test and retain project KBs


In [ ]:
prepare = PIPELINE_DIR / "prepare_dataset.py"
run([
    sys.executable, "-u", prepare,
    "--stage", "finalize",
    "--typepro-root", REPO_DIR,
    "--work-dir", MERGED_BUILD,
    "--output-dir", FINAL_DIR,
    "--split-profile", "paper_project",
    "--test-projects", 100,
    "--validation-project-ratio", 0.10,
    "--seed", SEED,
    "--preview-samples", 2,
    "--preview-max-chars", 1600,
    "--log-every", 10000,
])
run([sys.executable, PIPELINE_DIR / "verify_dataset.py", "--data-dir", FINAL_DIR])


## Display exact counts and examples


In [ ]:
manifest = json.loads((FINAL_DIR / "manifest.json").read_text(encoding="utf-8"))
stats = json.loads((FINAL_DIR / "preprocess_stats.json").read_text(encoding="utf-8"))
print(json.dumps({
    "output": manifest["output"],
    "prepared_counts": manifest["split"]["prepared_counts"],
    "prepared_projects": manifest["split"]["prepared_projects"],
    "preprocess_stats": stats,
    "project_knowledge_bases": manifest["projects"]["knowledge_bases"],
}, indent=2, ensure_ascii=False))
for split in ("train", "validation", "test"):
    print(f"\n===== {split.upper()} SAMPLES =====")
    with (FINAL_DIR / f"{split}.jsonl").open(encoding="utf-8") as handle:
        for _, line in zip(range(2), handle):
            print(json.dumps(json.loads(line), indent=2, ensure_ascii=False)[:3000])


## Publish final private Kaggle Dataset


In [ ]:
use_credential(FINAL_SOURCE)
final_id = f"{FINAL_SOURCE['owner']}/{FINAL_DATASET_SLUG}"
run([
    sys.executable, PIPELINE_DIR / "publish_kaggle.py",
    "--data-dir", FINAL_DIR,
    "--dataset-id", final_id,
    "--title", "TypePro Python Generative Project-KB Data",
    "--message", f"Merge {len(MERGE_DATASETS)} verified partitions covering {SHARD_COUNT} shards",
])
completion = {
    "dataset_id": final_id,
    "shard_count": SHARD_COUNT,
    "source_datasets": [item["dataset_id"] for item in MERGE_DATASETS],
    "output": manifest["output"],
}
(FINAL_DIR / "MERGE_COMPLETE.json").write_text(
    json.dumps(completion, indent=2), encoding="utf-8"
)
print(json.dumps(completion, indent=2))
